In [14]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [8]:
load_dotenv()
# Groq AI Model
model = ChatGroq(
  model = "openai/gpt-oss-20b"
)

In [9]:
# create a state

class LLMState(TypedDict):

  question: str
  answer: str
  

In [11]:
def llm_qa(state: LLMState)->LLMState:
  
  # extract the question from state
  question = state['question']

  # make a prompt
  prompt = f'Answer the following {question}'

  # ask the question to llm
  answer = model.invoke(prompt).content

  # update the answer in the state
  state['answer'] = answer

  return state

In [16]:
# create a graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [19]:
# execute

initial_state = {'question': 'How we should i write .gitignore in order to ignore the .env file?'}
final_output = workflow.invoke(initial_state)
print(final_output)

{'question': 'How we should i write .gitignore in order to ignore the .env file?', 'answer': '## Ignoring the `.env` file in Git\n\nTo keep your environment variables out of the repository, simply add a rule to your `.gitignore` that matches the file.  \nIf the file lives in the root of the repository, the rule is straightforward:\n\n```\n# .gitignore\n.env\n```\n\nThat single line tells Git to ignore **any file literally named `.env`** in the root directory.\n\n### Common variations\n\n| Situation | What to add to `.gitignore` | Notes |\n|-----------|---------------------------|-------|\n| **Only the root `.env`** | `.env` | Works only for the root; sub‑folders are unaffected. |\n| **All `.env` files anywhere** | `**/.env` | Uses the glob pattern `**/` to match any directory depth. |\n| **Specific extensions** | `.env*` | Ignores `.env`, `.env.local`, `.env.production`, etc. |\n| **Exclude from a particular folder** | `!/config/.env` | If you want to keep a `.env` in `config/` but ign